# 03 — Local Volatility via Dupire

The implied vol surface tells us *what* the market prices — but not *how* the spot evolves.
Dupire (1994) asked: given all European option prices, what is the unique diffusion
$dS = S[r\,dt + \sigma_{\text{loc}}(S,t)\,dW]$ consistent with these prices?

The answer is the **local volatility function**:

$$\sigma^2_{\text{loc}}(K, T) = \frac{\partial C/\partial T + (r-q)K\,\partial C/\partial K + qC}{\frac{1}{2}K^2\,\partial^2 C/\partial K^2}$$

This is remarkable: a *single* function $\sigma_{\text{loc}}(K,T)$ generates the entire implied vol surface.

**Practical notes:**
- We differentiate the *fitted implied vol spline*, not raw market prices (avoids noise amplification)
- The local vol is only well-defined where $\partial^2 C/\partial K^2 > 0$ (i.e. no butterfly arbitrage)
- At the wings, numerical noise can cause negative local variance — we return NaN in those regions

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt

from src.surface import VolSurface
from src.local_vol import LocalVolSurface

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.dpi"] = 120

## 1. Build the Implied Vol Surface (from Notebook 02)

In [ ]:
surf = VolSurface(
    ticker="SPY",
    q=0.013,
    min_open_interest=10,    # relaxed from default of 100 to ensure data
    min_volume=0,            # relaxed from default of 10; needed when running outside US market hours
)
surf.fit(n_moneyness=35, n_tenors=18)
print(f"Surface fitted. Spot = {surf.spot:.2f}")

## 2. Compute the Local Vol Surface

In [ ]:
lv = LocalVolSurface(surf)

# Spot-check ATM 3M local vol vs implied vol
T_3m = 90 / 365
iv_atm_3m = surf.iv(surf.spot, T_3m)
lv_atm_3m = lv.local_vol(surf.spot, T_3m)

print(f"ATM 3M Implied Vol: {iv_atm_3m:.2%}")
print(f"ATM 3M Local Vol:   {lv_atm_3m:.2%}")
print("\nNote: LV is close to IV at ATM, but they diverge significantly in the wings.")

## 3. Implied Vol vs Local Vol — Smile Comparison

A key result from Dupire theory: the local vol smile is **steeper** than the implied vol smile.
Intuitively, if the implied vol is high for low strikes, the local vol must be *even higher*
in that region to generate the observed market prices through path-averaging.

This has a practical implication: hedging with local vol produces different delta/gamma
than hedging with a sticky-strike or sticky-moneyness assumption.

In [ ]:
for T_days in [30, 90, 180]:
    lv.compare_iv_lv(
        T_target=T_days/365,
        save_path=f"../figures/03_iv_vs_lv_{T_days}d.png"
    )

## 4. Local Vol Surface — 3D Plot

In [ ]:
lv.plot_local_vol_surface(
    n_strikes=20,
    n_tenors=10,
    save_path="../figures/03_lv_surface_3d.png"
)

## 5. Cross-Sectional Local Vol vs Implied Vol

We summarise the IV vs LV comparison across all available tenors.
The ratio LV/IV is typically > 1 in the put wing and < 1 in the call wing.

In [ ]:
tenors_days = [30, 60, 90, 180]
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle("Implied Vol vs Local Vol Across Tenors", fontsize=13)

# Restrict to liquid moneyness range only
moneyness_min, moneyness_max = 0.85, 1.10

for ax, T_days in zip(axes.flat, tenors_days):
    T = T_days / 365
    strikes_full, iv_full = surf.smile(T, n_points=80)
    
    # Filter to liquid range
    moneyness_full = strikes_full / surf.spot
    mask = (moneyness_full >= moneyness_min) & (moneyness_full <= moneyness_max)
    strikes = strikes_full[mask]
    iv_vals = iv_full[mask]
    moneyness = strikes / surf.spot
    
    # Compute LV only on the same restricted range
    lv_vals = np.array([lv.local_vol(K, T) for K in strikes])

    ax.plot(moneyness, iv_vals * 100, label="Implied Vol", lw=2, color="#2196F3")
    ax.plot(moneyness, lv_vals * 100, label="Local Vol", lw=2, ls="--", color="#F44336")
    ax.axvline(1.0, color="grey", ls=":", alpha=0.5)
    ax.set_title(f"{T_days}d Tenor")
    ax.set_xlabel("Moneyness (K/S)")
    ax.set_ylabel("Vol (%)")
    ax.set_ylim(5, 40)  # clip to reasonable range
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../figures/03_iv_vs_lv_all_tenors.png", dpi=150, bbox_inches="tight")
plt.show()

## Summary

- Dupire local vol is the unique diffusion consistent with the full implied vol surface.
- The local vol smile is steeper than the implied vol smile — a direct consequence of path-averaging.
- In the wings, numerical differentiation becomes noisy; this is a known limitation of the local vol approach.
- More sophisticated models (Heston, SABR) address some of these issues at the cost of additional parameters.

**Next:** Notebook 04 — does the implied vol term structure predict future realised vol?